# Week 4 · Task 1.1 / 1.2 follow-up — Per-bin significance tests

The size-stratified (Task 1.1) and contrast-stratified (Task 1.2) tables are descriptive means. This notebook adds the paired bootstrap confidence intervals and Wilcoxon tests per bin, using the same conventions as Sebastian's `src/stats_analysis.py`.

**Conventions (same as the shared protocol).**
- Unit of analysis: the test image. The U-Net score of an image in a bracket is the mean over the 3 seeds; it is paired with the single SAM 3 score for the same image.
- Effect = **SAM 3 advantage**: `SAM 3 - U-Net` for Dice, IoU and Boundary IoU (higher is better); `U-Net - SAM 3` for HD95 (lower is better). Positive therefore always means SAM 3 is better.
- 95 % CI: percentile bootstrap over images, 10,000 resamples (`configs/eval.yaml`), fixed seed. Image resampling only: the seed stage of the two-stage bootstrap is not repeated here (3 seeds are averaged first).
- Test: two-sided Wilcoxon signed-rank on the paired differences. Rank-biserial `r` is the effect size (positive = SAM 3 better).
- HD95: non-finite values capped at the largest finite value in the data (policy logged in `decisions.md`, 21 Oct); summary = median of paired differences.
- Multiple comparisons: p-values are adjusted within each metric across all bins, brackets and strategies, with both Holm and Bonferroni. Quote Holm unless the group agrees otherwise. Size bins are the pre-declared analysis; contrast bins are exploratory.
- Strategies tested: `box-gt` (oracle), `box-j10` (realistic box), `pt-3`. Fewer strategies keeps the number of tests, and the multiplicity penalty, manageable.

**Check.** The cell below reproduces Sebastian's all-image Wilcoxon results before any bin is analysed.

**Power.** The small bin has only 39 images, so wide intervals and non-significant results there are expected and should be reported as inconclusive, not as "no difference".

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "results" / "week4"
OUT.mkdir(parents=True, exist_ok=True)

N_BOOT = 10_000
BOOT_SEED = 20260101
STRATEGIES = ["box-gt", "box-j10", "pt-3"]
BRACKETS = ["10pct", "25pct", "50pct", "100pct"]
METRICS = {"dice": +1, "boundary_iou": +1, "hd95": -1}     # direction: +1 higher is better, -1 lower is better
SIZE_ORDER = ["small", "medium", "large"]
CONTRAST_ORDER = ["low", "medium", "high"]

## 1. Build the paired per-image table

In [ ]:
df = pd.read_csv(ROOT / "results" / "per_image_results.csv")
df["hd95"] = df["hd95"].replace([np.inf, -np.inf], np.nan)
finite_max = df["hd95"].max()
n_nonfinite = int(df["hd95"].isna().sum())
df["hd95"] = df["hd95"].fillna(finite_max)
print(f"HD95: {n_nonfinite} non-finite rows capped at {finite_max:.3f}")

df["bracket"] = df.run_id.str.extract(r"^(.+?)[_-]seed\d+$")[0]
# SAM 3 scores: one per image and strategy
sam = {m: df[df.arm == "sam3"].pivot(index="image_id", columns="run_id", values=m) for m in METRICS}
# U-Net scores: mean over seeds per image and bracket
unet = {m: df[df.arm == "unet"].groupby(["image_id", "bracket"])[m].mean().unstack() for m in METRICS}
images = sorted(sam["dice"].index)
assert len(images) == 200 and all(unet[m].shape == (200, 4) for m in METRICS)

size = pd.read_csv(ROOT / "results" / "week4" / "size_bin_assignment.csv").set_index("image_id")["size_bin"]
contrast = pd.read_csv(ROOT / "results" / "week4" / "contrast_per_image.csv").set_index("image_id")["contrast_bin"]
print(size.value_counts().reindex(SIZE_ORDER).to_dict(), contrast.value_counts().reindex(CONTRAST_ORDER).to_dict())

## 2. Test function

In [ ]:
def advantage(metric, bracket, strategy, ids):
    s = sam[metric].loc[ids, strategy].to_numpy(float)
    u = unet[metric].loc[ids, bracket].to_numpy(float)
    return (s - u) if METRICS[metric] > 0 else (u - s)       # positive = SAM 3 better

def summarise(d, metric, rng):
    stat = np.median if metric == "hd95" else np.mean
    est = stat(d)
    idx = rng.integers(0, len(d), size=(N_BOOT, len(d)))
    boots = stat(d[idx], axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    nz = d[d != 0]
    if len(nz) == 0:
        p, r = 1.0, 0.0
    else:
        p = stats.wilcoxon(d, zero_method="wilcox", alternative="two-sided").pvalue
        ranks = stats.rankdata(np.abs(nz))
        wp, wn = ranks[nz > 0].sum(), ranks[nz < 0].sum()
        r = (wp - wn) / (wp + wn)
    return est, lo, hi, p, r, float((d < 0).mean())     # last = share of images where U-Net is better

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

## 3. Check against Sebastian's all-image tests

For the unstratified 200 images, my Wilcoxon p-values, effect sizes and median differences must match `results/analysis/wilcoxon.csv`. His `median_diff` is U-Net minus SAM 3 for Dice, so the sign is flipped here.

In [ ]:
rng = np.random.default_rng(BOOT_SEED)
seb = pd.read_csv(ROOT / "results" / "analysis" / "wilcoxon.csv")
rows = []
for b in BRACKETS:
    for s in STRATEGIES:
        d = advantage("dice", b, s, images)
        est, lo, hi, p, r, _ = summarise(d, "dice", rng)
        ref = seb[(seb.metric == "dice") & (seb.unet_bracket == b) & (seb.sam3_strategy == s)].iloc[0]
        rows.append({"bracket": b, "strategy": s, "p_mine": p, "p_seb": ref.p_raw,
                     "median_adv_mine": np.median(d), "median_adv_seb(-diff)": -ref.median_diff,
                     "|r|_mine": abs(r), "|r|_seb": abs(ref.rank_biserial)})
chk = pd.DataFrame(rows)
ok_p = np.allclose(chk.p_mine, chk.p_seb, rtol=1e-6, atol=1e-12)
ok_m = np.allclose(chk["median_adv_mine"], chk["median_adv_seb(-diff)"], atol=1e-9)
ok_r = np.allclose(chk["|r|_mine"], chk["|r|_seb"], atol=1e-4)
print("p-values match:", ok_p, "| median differences match:", ok_m, "| effect sizes match:", ok_r)
chk

## 4. Size bins (pre-declared analysis)

In [ ]:
def run_bins(bin_series, order, label):
    rng = np.random.default_rng(BOOT_SEED)
    out = []
    for metric in METRICS:
        for bin_name in order:
            ids = [i for i in images if bin_series[i] == bin_name]
            for b in BRACKETS:
                for s in STRATEGIES:
                    d = advantage(metric, b, s, ids)
                    est, lo, hi, p, r, share = summarise(d, metric, rng)
                    out.append({"bin_type": label, "bin": bin_name, "n_images": len(ids), "metric": metric,
                                "bracket": b, "strategy": s, "sam3_advantage": est, "ci_lo": lo, "ci_hi": hi,
                                "p_raw": p, "rank_biserial": r, "share_unet_better": share})
    res = pd.DataFrame(out)
    res["p_holm"] = np.nan; res["p_bonferroni"] = np.nan
    for metric in METRICS:                                    # one family per metric, across bins x brackets x strategies
        m = res.metric == metric
        res.loc[m, "p_holm"] = holm(res.loc[m, "p_raw"])
        res.loc[m, "p_bonferroni"] = np.minimum(1.0, res.loc[m, "p_raw"] * m.sum())
    res["sig_holm"] = res.p_holm < 0.05
    res["ci_excludes_0"] = (res.ci_lo > 0) | (res.ci_hi < 0)
    return res

size_res = run_bins(size, SIZE_ORDER, "size")
size_res.to_csv(OUT / "size_bin_significance.csv", index=False)
print("tests per metric:", int((size_res.metric == "dice").sum()))
size_res.groupby(["metric"]).agg(n_tests=("p_raw", "size"), sig_holm=("sig_holm", "sum"), ci_excl0=("ci_excludes_0", "sum"))

In [ ]:
def show(res, metric, strategies=STRATEGIES, fmt="{:+.3f}"):
    r = res[res.metric == metric].copy()
    star = np.where(r.sig_holm, "*", "")
    r["cell"] = [f"{fmt.format(e)} [{fmt.format(l)}, {fmt.format(h)}]{s}" for e, l, h, s in zip(r.sam3_advantage, r.ci_lo, r.ci_hi, star)]
    t = r.pivot_table(index=["strategy", "bracket"], columns="bin", values="cell", aggfunc="first")
    t = t[[c for c in (SIZE_ORDER if "small" in t.columns else CONTRAST_ORDER)]]
    t.index = pd.MultiIndex.from_tuples([(s, b) for s, b in t.index])
    order = [(s, b) for s in strategies for b in BRACKETS]
    return t.loc[order]

print("Dice: SAM 3 advantage [95% CI], positive = SAM 3 better; * = Holm-adjusted p < 0.05")
show(size_res, "dice")

In [ ]:
print("Boundary IoU: SAM 3 advantage [95% CI]; * = Holm p < 0.05")
display(show(size_res, "boundary_iou"))
print("HD95 (px, median of paired differences, U-Net minus SAM 3, so positive = SAM 3 better); * = Holm p < 0.05")
display(show(size_res, "hd95", fmt="{:+.1f}"))

## 5. Figure: Dice advantage by size bin

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
xs = np.arange(len(BRACKETS))
w = 0.26
cols = {"small": "tab:red", "medium": "tab:orange", "large": "tab:blue"}
for ax, s in zip(axes, STRATEGIES):
    for k, bn in enumerate(SIZE_ORDER):
        r = size_res[(size_res.metric == "dice") & (size_res.strategy == s) & (size_res.bin == bn)].set_index("bracket").loc[BRACKETS]
        ax.errorbar(xs + (k - 1) * w, r.sam3_advantage, yerr=[r.sam3_advantage - r.ci_lo, r.ci_hi - r.sam3_advantage],
                    fmt="o", color=cols[bn], capsize=3, label=f"{bn} (n={int(r.n_images.iloc[0])})")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(xs); ax.set_xticklabels([b.replace("pct", "%") for b in BRACKETS])
    ax.set_title(f"SAM 3 {s}"); ax.set_xlabel("U-Net training data"); ax.grid(alpha=0.3)
axes[0].set_ylabel("SAM 3 Dice minus U-Net Dice\n(positive = SAM 3 better)")
axes[0].legend(fontsize=8, title="polyp size")
fig.suptitle("Dice advantage of SAM 3 over the U-Net by polyp size (95% bootstrap CI)", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "size_bin_dice_advantage.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Contrast tertiles (exploratory)

Same tests on the low / medium / high contrast tertiles from Task 1.2. These are exploratory: the proxy is confounded with polyp size and failed a face-validity check (see `decisions.md`, 20 Oct). Do not make claims from this table without saying so.

In [ ]:
contrast_res = run_bins(contrast, CONTRAST_ORDER, "contrast")
contrast_res.to_csv(OUT / "contrast_bin_significance.csv", index=False)
print("Boundary IoU by contrast tertile (exploratory): SAM 3 advantage [95% CI]; * = Holm p < 0.05")
show(contrast_res, "boundary_iou")

## 7. Notes for the report (fill in after reading the tables)

- **Size.** For which size bins, brackets and prompts is SAM 3's Dice advantage significant (Holm), and where does the sign flip? Use the CI, not only the star.
- **Small bin.** Only 39 images: say which comparisons are inconclusive.
- **Boundary metrics.** Does the pattern for Boundary IoU and HD95 match Dice, or do crossovers move?
- **Contrast.** Exploratory only: state the confound and the lack of support for H2 in the predicted direction.
- **Multiplicity.** p-values are adjusted within each metric across all bins, brackets and strategies (Holm and Bonferroni both saved).